# Explore evaluation results

Loads a run from `eval/results/` and lets you inspect scores, failures, retrieved evidence and
answers. Run from the project root (`assessment/`).

Sections:
1. Load a run
2. Headline scores
3. Per-question table
4. Breakdowns (type / source / expected status)
5. Drill into one question
6. Evidence check — which required quote was missed
7. Re-ask live (calls the API)
8. Compare two runs

In [3]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
sys.path.insert(0, str(ROOT))

from eval.evidence import group_hit, quote_in  # noqa: E402

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

RESULTS_DIR = ROOT / "eval" / "results"
runs = sorted(p for p in RESULTS_DIR.iterdir() if (p / "run.json").exists())
print("available runs:", [p.name for p in runs])

RUN = runs[-1]  # <- or a run name, e.g. "01-baseline-k5"
print("using:", RUN.name)

available runs: ['20260917-233403']
using: 20260917-233403


In [4]:
from eval import report  # noqa: E402

run = report.load_run(RUN)
results, summary, answers, golden = run.results, run.summary(), run.answers, run.golden

METRICS = report.METRICS
JUDGED = report.JUDGED

print(json.dumps(run.config, indent=2))
print(f"{len(results)} questions, mean latency {results['latency_s'].mean():.1f}s")

{
  "llm_model": "gemini/gemini-3.5-flash",
  "embed_model": "gemini/gemini-embedding-001",
  "judge_model": "gemini/gemini-3.5-flash",
  "top_k": 5,
  "max_chunk_words": 300
}
20 questions, mean latency 5.2s


## 2. Headline scores

`pass` applies the gates in `eval/scoring.py`, which depend on the expected support status.
Judged metrics are only computed where they apply (e.g. `missing_points_named` for partial
questions); questions that should be abstained on are judged by exact abstention instead.

In [5]:
pd.DataFrame([summary["overall"]]).T.rename(columns={0: "score"})

,score
status_correct,1.000
evidence_recall,0.971
rule_pass,0.950
faithfulness,0.821
context_precision,0.937
context_recall,0.957
factual_correctness,0.608
n,20.000


## 3. Per-question table

In [6]:
view = ["id", "source", "type", "expected_status", "predicted_status", *METRICS, "fail_reasons"]
results[view].style.format(precision=2).highlight_between(subset=JUDGED, right=0.79, props="font-weight:bold;color:#b00")

,id,source,type,expected_status,predicted_status,status_correct,evidence_recall,rule_pass,faithfulness,context_precision,context_recall,factual_correctness
0,S01,provided,multi-passage,supported,supported,True,1.00,True,1.00,1.00,1.00,0.67
1,S02,provided,misleading,supported,supported,True,1.00,True,0.83,1.00,0.60,0.60
2,S03,provided,answerable,supported,supported,True,1.00,True,1.00,1.00,1.00,0.50
3,S04,provided,answerable,supported,supported,True,1.00,True,0.25,1.00,1.00,0.75
4,S05,provided,answerable,supported,supported,True,1.00,True,1.00,1.00,1.00,1.00
5,S06,provided,answerable,supported,supported,True,1.00,True,1.00,1.00,1.00,1.00
6,S07,provided,misleading,supported,supported,True,1.00,True,0.80,1.00,1.00,0.55
7,S08,provided,partial,partially supported,partially supported,True,1.00,True,0.50,1.00,1.00,0.40
8,S09,provided,partial,partially supported,partially supported,True,1.00,True,1.00,0.76,1.00,0.25
9,S10,provided,adversarial,not supported,not supported,True,nan,True,nan,nan,nan,nan


In [7]:
# Gate failures and why.
failed = results[results["pass"] == False]  # noqa: E712
print("failed:", list(failed["id"]))
failed[view]

rule failures: ['C04']
weak judged scores: ['S04', 'S08', 'S09', 'C01', 'C02', 'C04', 'C05', 'C09']


,id,source,type,expected_status,predicted_status,status_correct,evidence_recall,rule_pass,faithfulness,context_precision,context_recall,factual_correctness
13,C04,self-generated,multi-passage,supported,supported,True,0.5,False,0.400,0.833,0.667,0.36
3,S04,provided,answerable,supported,supported,True,1.0,True,0.250,1.000,1.000,0.75
7,S08,provided,partial,partially supported,partially supported,True,1.0,True,0.500,1.000,1.000,0.40
8,S09,provided,partial,partially supported,partially supported,True,1.0,True,1.000,0.756,1.000,0.25
10,C01,self-generated,answerable,supported,supported,True,1.0,True,0.667,1.000,1.000,1.00
11,C02,self-generated,multi-passage,supported,supported,True,1.0,True,1.000,0.589,1.000,0.17
14,C05,self-generated,partial,partially supported,partially supported,True,1.0,True,0.500,1.000,1.000,0.40
18,C09,self-generated,misleading,supported,supported,True,1.0,True,1.000,0.833,1.000,0.40


## 4. Breakdowns

In [8]:
for key in ["by_type", "by_source", "by_expected_status"]:
    print(f"--- {key}")
    display(pd.DataFrame(summary[key]).T)

--- by_type


,status_correct,evidence_recall,rule_pass,faithfulness,context_precision,context_recall,factual_correctness,n
adversarial,1.0,1.000,1.00,1.000,1.000,1.000,0.670,2.0
ambiguous,1.0,1.000,1.00,1.000,1.000,1.000,0.800,1.0
answerable,1.0,1.000,1.00,0.783,1.000,1.000,0.850,5.0
misleading,1.0,1.000,1.00,0.878,0.944,0.867,0.517,3.0
multi-passage,1.0,0.875,0.75,0.850,0.835,0.917,0.505,4.0
partial,1.0,1.000,1.00,0.667,0.919,1.000,0.350,3.0
unsupported,1.0,NaN,1.00,NaN,NaN,NaN,NaN,2.0


--- by_source


,status_correct,evidence_recall,rule_pass,faithfulness,context_precision,context_recall,factual_correctness,n
provided,1.0,1.000,1.0,0.820,0.973,0.956,0.636,10.0
self-generated,1.0,0.938,0.9,0.821,0.897,0.958,0.578,10.0


--- by_expected_status


,status_correct,evidence_recall,rule_pass,faithfulness,context_precision,context_recall,factual_correctness,n
not supported,1.0,NaN,1.000,NaN,NaN,NaN,NaN,3.0
partially supported,1.0,1.000,1.000,0.667,0.919,1.000,0.350,3.0
supported,1.0,0.964,0.929,0.854,0.941,0.948,0.664,14.0


## 5. Drill into one question

`show("C04")` prints the expected behaviour, the answer, the scores, and every retrieved passage.
`*` marks a passage the answer cited.

In [9]:
def show(qid: str, chars: int = 400) -> None:
    item, answer = golden[qid], answers[qid]
    row = results[results["id"] == qid].iloc[0]
    print(f"{qid} [{item['source']} / {item['type']}]\n\nQ: {item['question']}\n")
    print(f"expected: {item['expected_status']}   predicted: {answer['support_status']}")
    print("scores:", {m: row[m] for m in METRICS})
    if isinstance(row["fail_reasons"], str):
        print("failed because:", row["fail_reasons"])
    if item.get("missing_points"):
        print("expected missing:", item["missing_points"])
    if answer["missing_information"]:
        print("reported missing:", answer["missing_information"])
    if answer["warnings"]:
        print("warnings:", answer["warnings"])
    print(f"\nREFERENCE:\n{item['reference']}\n\nANSWER:\n{answer['answer']}\n")
    print("RETRIEVED:")
    for c in answer["retrieved"]:
        mark = "*" if c["chunk_id"] in answer["citations"] else " "
        print(f" {mark} [{c['chunk_id']}] {c['section']} (distance={c['distance']:.3f})")
        print("     ", " ".join(c["text"].split())[:chars], "...")


show("C04")

C04 [self-generated / multi-passage]

Q: A de-identified patient dataset was breached, but the identity mapping table was not. Does the organisation have to notify anyone, and what determines that?

expected: supported   predicted: supported
scores: {'status_correct': np.True_, 'evidence_recall': np.float64(0.5), 'rule_pass': np.False_, 'faithfulness': np.float64(0.4), 'context_precision': np.float64(0.833), 'context_recall': np.float64(0.667), 'factual_correctness': np.float64(0.36)}

REFERENCE:
The organisation must assess whether the breach is notifiable, because de-identified data has a higher re-identification risk. A data breach is generally notifiable if it results in, or is likely to result in, significant harm to affected individuals, or is of significant scale. If the breach is notifiable, the organisation must notify affected individuals and/or the Commission.

ANSWER:
If de-identified data alone is breached (while the identity mapping table remains secure), the organisation

## 6. Evidence check

Which required evidence group was missed, and which quote would have satisfied it.

In [10]:
class _Chunk:  # group_hit() only needs .source and .text
    def __init__(self, d):
        self.source, self.text = d["source"], d["text"]


def evidence_report(qid: str) -> pd.DataFrame:
    item, answer = golden[qid], answers[qid]
    retrieved = [_Chunk(c) for c in answer["retrieved"]]
    cited = [_Chunk(c) for c in answer["retrieved"] if c["chunk_id"] in answer["citations"]]
    rows = []
    for i, group in enumerate(item["required_evidence"], 1):
        rows.append(
            {
                "group": i,
                "retrieved": group_hit(group, retrieved),
                "cited": group_hit(group, cited),
                "quotes": " || ".join(ev["quote"][:80] for ev in group),
            }
        )
    return pd.DataFrame(rows)


# Every question with a missed group
for qid in golden:
    if qid in answers and golden[qid]["required_evidence"]:
        groups = evidence_report(qid)
        if not groups["retrieved"].all():
            print(f"--- {qid}")
            display(groups)

--- C04


,group,retrieved,cited,quotes
0,1,True,True,"If de-identified data alone is breached, the organisation must assess whether th"
1,2,False,False,"results in, or is likely to result in, significant harm to affected individuals"


In [11]:
# Where does a missing quote actually live? Useful for deciding top-k / hybrid retrieval.
from rag.chunking import load_chunks  # noqa: E402
from rag.config import get_settings  # noqa: E402

chunks = load_chunks(get_settings().docs_dir)


def locate(quote: str) -> list[str]:
    return [f"{c.chunk_id} :: {c.section}" for c in chunks if quote_in(quote, c.text)]


locate("results in, or is likely to result in, significant harm to affected individuals")

['kc-25 :: 24. Data Breach Notification Obligation']

## 7. Re-ask live

Calls the API. Useful for checking whether a prompt or `top_k` change fixes a failing case.

In [12]:
from rag.pipeline import RagPipeline  # noqa: E402

pipeline = RagPipeline()

qid = "C04"
result = pipeline.ask(golden[qid]["question"], k=8)  # try a larger k
print(result.support_status, result.citations)
print(result.answer)
print()
retrieved = [_Chunk({"source": c.source, "text": c.text}) for c in result.retrieved]
for i, group in enumerate(golden[qid]["required_evidence"], 1):
    print(f"group {i} retrieved:", group_hit(group, retrieved))

supported ['anon-30', 'policy-08', 'kc-25', 'hc-27']
If a de-identified dataset alone is breached, the organisation must assess whether the breach is notifiable, as de-identified data carries a higher re-identification risk [anon-30]. Under internal policy, de-identified patient-level data must be treated as sensitive unless a re-identification risk assessment supports a lower-risk classification [policy-08].

If the breach is assessed to be notifiable, the organisation must notify the affected individuals and/or the Commission [anon-30, kc-25, hc-27]. Whether a breach is determined to be notifiable depends on whether it:
1. Results in, or is likely to result in, significant harm to affected individuals; or
2. Is of significant scale [kc-25].

group 1 retrieved: True
group 2 retrieved: True


## 8. Compare two runs

In [ ]:
def compare(run_a: Path, run_b: Path) -> pd.DataFrame:
    cols = METRICS
    a = report.load_run(run_a).results.set_index("id")[cols].astype(float)
    b = report.load_run(run_b).results.set_index("id")[cols].astype(float)
    return pd.concat({run_a.name: a, run_b.name: b, "delta": b - a}, axis=1)


if len(runs) > 1:
    display(compare(runs[-2], runs[-1]))
else:
    print("only one run so far")